# Nexus_v1 — Week-1 Kill Experiment (reward-free boundary discovery)

Loads a **frozen, pretrained EMERALD** checkpoint from Drive and runs the §16.1 kill experiment:

> frozen EMERALD → collect replay → train **only** the skill tier on its latents (λ_r = 0) → streaming-restart segmentation → **look at the cuts**.

This tests the one load-bearing assumption (A2 / H-RF): *do dynamics funnels coincide with task structure?* EMERALD is **never retrained** — only `skill_tier_parameters()` get gradients.

**Prereq:** you have already trained EMERALD (the other notebook) and `latest.pt` is on Drive at `MyDrive/nexus/emerald_smoke/`.

**Outputs** (to `MyDrive/nexus/kill_week1/`): `overlay_*.png` (cuts vs achievement unlocks) + `verdict.json` (degeneracy sweep over ℓ̄∈{25,50,100} + boundary↔achievement F1).

In [ ]:
# 1. Confirm the GPU
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2. Get the code (kill_experiment.py lives in nexus/)
%cd /content
!git clone https://github.com/JuliusSamwer/Nexus_v1.git 2>/dev/null || (cd Nexus_v1 && git pull -q)
%cd /content/Nexus_v1
!pip install -q crafter matplotlib
print('ready')

In [ ]:
# 3. Mount Drive and locate the trained EMERALD checkpoint
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT = '/content/drive/MyDrive/nexus/emerald_smoke/latest.pt'
OUT  = '/content/drive/MyDrive/nexus/kill_week1'
os.makedirs(OUT, exist_ok=True)
assert os.path.exists(CKPT), f'Checkpoint not found: {CKPT}\nRun the EMERALD training notebook first.'
import torch
_ck = torch.load(CKPT, map_location='cpu')
print('checkpoint OK | trained to step', _ck.get('step', '?'),
      '| params', sum(v.numel() for v in _ck['agent'].values()))
del _ck

In [ ]:
# 4. Run the kill experiment (frozen EMERALD; only the skill tier trains; λ_r = 0)
#    If it errors 'No episode reaches T=128', lower --window to 96.
!python3 -m nexus.kill_experiment \
    --checkpoint "{CKPT}" --out "{OUT}" --device cuda --configs base \
    --collect_eps 64 --skill_steps 1500 --window 128 \
    --batch_hl 8 --n_overlay 6 --tol 5 --lambda_r 0.0

In [ ]:
# 5. The verdict (§14 row-1/row-2): degeneracy sweep + boundary↔achievement F1
import json
with open(os.path.join(OUT, 'verdict.json')) as f:
    verdict = json.load(f)
print(json.dumps(verdict, indent=2))

f1 = verdict['boundary_achievement_f1']['mean']
deg = verdict['degeneracy_sweep_ell_bar']
degenerate = all(d['frac_len1'] > 0.9 or d['frac_ge_Lmax'] > 0.9 for d in deg.values())
print('\n================ READ ================')
print(f'boundary↔achievement F1 (mean)   : {f1}')
print(f'segmentation degenerate?         : {degenerate}  (True = bad: all length-1 or one-giant)')
print('GREEN if not-degenerate AND cuts visibly track unlocks in the overlays below.')

In [ ]:
# 6. LOOK AT THE CUTS — discovered boundaries (blue) vs achievement unlocks (red dashed)
from IPython.display import Image, display
import glob
for png in sorted(glob.glob(os.path.join(OUT, 'overlay_*.png'))):
    print(os.path.basename(png))
    display(Image(filename=png))